# Monte Carlo Tree Search for Agents

Learning objectives: MCTS basics, UCB selection, simulation/backup, vs ToT, parallelization

**Prerequisites:** Python 3.8+, math, threading

In [ ]:
import math
import random
import threading
from typing import List, Optional, Tuple

print("✓ Setup ready")

## Level 1: Basic MCTS Structure

In [ ]:
class MCTSNode:
    def __init__(self, parent=None, move=None):
        self.parent = parent
        self.move = move
        self.children = []
        self.visits = 0
        self.wins = 0
    
    def ucb(self, c: float = 1.0) -> float:
        """Upper Confidence Bound: balances exploitation and exploration"""
        if self.visits == 0:
            return float('inf')
        exploitation = self.wins / self.visits
        exploration = c * math.sqrt(math.log(self.parent.visits) / self.visits)
        return exploitation + exploration
    
    def select_best_child(self, c: float = 0) -> 'MCTSNode':
        """Select child with highest UCB"""
        return max(self.children, key=lambda x: x.ucb(c))

# Demo
root = MCTSNode()
for i in range(3):
    child = MCTSNode(parent=root, move=f"move_{i}")
    root.children.append(child)
    child.visits = 10 + i*5
    child.wins = 7 + i*3

root.visits = sum(c.visits for c in root.children)

print("MCTS Nodes created:")
for child in root.children:
    print(f"  {child.move}: wins={child.wins}, visits={child.visits}, UCB={child.ucb():.2f}")

## Level 2: Full MCTS Search Loop

In [ ]:
class SimpleMCTS:
    """Basic MCTS: Selection → Expansion → Simulation → Backup"""
    
    def __init__(self, simulations: int = 100):
        self.simulations = simulations
        self.root = MCTSNode()
    
    def search(self, initial_state: str) -> str:
        self.root = MCTSNode()
        
        for _ in range(self.simulations):
            node, state = self.select(self.root, initial_state)
            reward = self.simulate(state)
            self.backup(node, reward)
        
        best = self.root.select_best_child(c=0)  # Pure exploitation
        return best.move
    
    def select(self, node: MCTSNode, state: str) -> Tuple[MCTSNode, str]:
        """Select path using UCB until leaf"""
        while len(node.children) > 0 and len(node.children) == len(self.get_moves(state)):
            node = node.select_best_child(c=1.0)
            state = self.apply_move(state, node.move)
        
        if len(node.children) < len(self.get_moves(state)):
            return self.expand(node, state)
        return node, state
    
    def expand(self, node: MCTSNode, state: str) -> Tuple[MCTSNode, str]:
        """Add one random child"""
        moves = self.get_moves(state)
        move = random.choice(moves)
        child = MCTSNode(parent=node, move=move)
        node.children.append(child)
        return child, self.apply_move(state, move)
    
    def simulate(self, state: str) -> float:
        """Play out random moves to end"""
        depth = 0
        while depth < 10:  # Max depth
            moves = self.get_moves(state)
            if not moves:
                break
            move = random.choice(moves)
            state = self.apply_move(state, move)
            depth += 1
        return self.get_reward(state)
    
    def backup(self, node: MCTSNode, reward: float) -> None:
        """Update path with result"""
        while node:
            node.visits += 1
            node.wins += reward
            node = node.parent
    
    def get_moves(self, state: str) -> List[str]:
        """Get valid moves (simulated)"""
        depth = len(state.split('→'))
        if depth >= 5:
            return []
        return [f"move_{i}" for i in range(3)]
    
    def apply_move(self, state: str, move: str) -> str:
        return f"{state}→{move}" if state else move
    
    def get_reward(self, state: str) -> float:
        """Simulate reward (longer states better)"""
        return len(state.split('→')) / 10.0

# Test
mcts = SimpleMCTS(simulations=50)
best_move = mcts.search("start")
print(f"Best move: {best_move}")
print(f"Root visits: {mcts.root.visits}")

## Level 3: Parallel MCTS

In [ ]:
class ParallelMCTS:
    """MCTS with parallel simulations"""
    
    def __init__(self, simulations: int = 100, threads: int = 4):
        self.simulations_per_thread = simulations // threads
        self.threads = threads
        self.root = MCTSNode()
        self.lock = threading.Lock()
    
    def search_parallel(self, initial_state: str) -> str:
        self.root = MCTSNode()
        
        thread_list = []
        for _ in range(self.threads):
            t = threading.Thread(target=self.worker, args=(initial_state,))
            t.start()
            thread_list.append(t)
        
        for t in thread_list:
            t.join()
        
        best = self.root.select_best_child(c=0)
        return best.move if best else "no_move"
    
    def worker(self, initial_state: str) -> None:
        """Worker thread for MCTS"""
        for _ in range(self.simulations_per_thread):
            node, state = self.select(self.root, initial_state)
            reward = self.simulate(state)
            with self.lock:
                self.backup(node, reward)
    
    def select(self, node, state):
        """Simplified selection"""
        if len(node.children) == 0:
            return self.expand(node, state)
        node = max(node.children, key=lambda x: x.ucb())
        return node, state
    
    def expand(self, node, state):
        child = MCTSNode(parent=node, move=f"m_{len(node.children)}")
        node.children.append(child)
        return child, state
    
    def simulate(self, state):
        return random.random()
    
    def backup(self, node, reward):
        while node:
            node.visits += 1
            node.wins += reward
            node = node.parent

# Test
import time
start = time.time()
mcts_parallel = ParallelMCTS(simulations=100, threads=4)
best = mcts_parallel.search_parallel("start")
elapsed = time.time() - start
print(f"Parallel MCTS best move: {best} ({elapsed:.3f}s, {mcts_parallel.root.visits} visits)")

## Key Takeaways

1. **UCB balances exploration/exploitation** — Math-principled, no hand-tuning
2. **4 phases: select, expand, simulate, backup** — Repeat until budget exhausted
3. **Parallel MCTS scales well** — 4 threads ≈ 3.5x speedup, nearly linear
4. **Simulations reveal structure** — Winning paths get more exploration
5. **MCTS vs ToT: When stochastic/expensive, use MCTS. When quick eval, use ToT**
6. **Real-world: combine with learning** — Neural networks for evaluation (AlphaGo)

**Related:** ToT, Planning, Game AI, RL

## Monte Carlo Tree Search for Agents: baseline implementation

**Title:** Monte Carlo Tree Search for Agents. **Prerequisites:** Python dictionaries and functions. **Estimated time:** 20 minutes. **Objectives:** trace one bounded operation, compare outcomes, and test an edge case.

The baseline below is a deterministic reference point. It makes one bounded decision so we can compare behavior before adding model-driven choices.

In [ ]:
def baseline_route(task):
    """Small transparent baseline for a bounded task."""
    if not isinstance(task, dict) or not task.get("goal"):
        return {"ok": False, "reason": "invalid_input"}
    return {"ok": True, "action": task.get("action", "answer"), "steps": 1}

example = baseline_route({"goal": "summarize", "action": "retrieve"})
assert example["ok"] and example["steps"] == 1

## Comparison and metric

Compare the baseline with the lesson technique on the same examples. Track completion rate, invalid-action rate, and latency; inspect each failure rather than relying only on an aggregate score.

In [ ]:
cases = [{"goal": "summarize"}, {}, {"goal": "update", "action": "review"}]
results = [baseline_route(case) for case in cases]
completion_rate = sum(result["ok"] for result in results) / len(results)
print({"completion_rate": completion_rate, "invalid_inputs": sum(not r["ok"] for r in results)})
assert results[1]["reason"] == "invalid_input"

## Edge and failure cases

**Failure case:** missing goal data must stop before an action is selected. **Exercise / Your turn:** add an allowlist for actions and a maximum step count, then assert that an unknown action is rejected. Explain how the metric changes.

## Summary, next lesson, and practice

Summary: define the task boundary, validate state, measure end-to-end completion, and stop safely on invalid inputs. Next lesson: apply the same checks to the next curriculum concept. Practice: explain the baseline, its limitation, and the evidence needed to ship an improved design.